# Mitsui Commodities Challenge 2025 – TimeXer Starter Notebook

This notebook provides a **minimal but complete** training & inference pipeline for the [Mitsui&Co. Commodity Prediction Challenge](https://www.kaggle.com/competitions/mitsui-commodity-prediction-challenge/overview).
Key components:
1. Data loading & basic preprocessing
2. Custom evaluation metric – *Sharpe‑like Spearman ratio*
3. TimeXer model definition & training loop
4. Validation using the official metric
5. Inference & submission file creation

*Adapt / extend each section to improve performance (feature engineering, larger models, richer data, etc.).*

In [ ]:
#! If running on Kaggle, many libs are pre‑installed.
#! Uncomment as needed.
#! pip install timexer neuralforecast


In [ ]:
import os, gc, sys, math, json, random, pathlib, itertools, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch import nn

# TimeXer – two options
# 1) The original THUML implementation
# from timexer import TimeXer
# 2) Nixtla's neuralforecast wrapper (easier)
from neuralforecast.models import TimeXer

warnings.filterwarnings('ignore')


In [ ]:
DATA_DIR = pathlib.Path('/kaggle/input/mitsui-commodity-prediction-challenge')
TRAIN_FILE = DATA_DIR/'train.parquet'
SERIES_FILE = DATA_DIR/'train_series.parquet'
ASSETS_FILE = DATA_DIR/'assets.csv'
TEST_DIR = DATA_DIR/'example_test_files'


In [ ]:
train_df = pd.read_parquet(TRAIN_FILE)
series_df = pd.read_parquet(SERIES_FILE)
assets_df = pd.read_csv(ASSETS_FILE)

print(train_df.shape, series_df.shape, assets_df.shape)

In [ ]:
# Merge metadata (optional)
train_df = train_df.merge(assets_df, on='target_id', how='left')

# Identify numeric feature columns
feature_cols = [c for c in train_df.columns if c.startswith('feature_')]

# Drop features with >90% NaN
thresh = int(0.9 * len(train_df))
train_df = train_df.dropna(axis=1, thresh=thresh)
feature_cols = [c for c in feature_cols if c in train_df.columns]

# Fill remaining NaNs (quick baseline: zero fill)
train_df[feature_cols] = train_df[feature_cols].fillna(0)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])


In [ ]:
SEQ_LEN = 64   # history length
PRED_LEN = 1   # one‑step ahead

def build_sequences(df, feature_cols, seq_len=SEQ_LEN, pred_len=PRED_LEN):
    X, y = [], []
    for tid, grp in tqdm(df.groupby('target_id')):
        grp = grp.sort_values('timestamp').reset_index(drop=True)
        feats = grp[feature_cols].values
        targets = grp['target'].values
        for i in range(len(grp) - seq_len - pred_len + 1):
            X.append(feats[i:i+seq_len])
            y.append(targets[i+seq_len:i+seq_len+pred_len])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

X, y = build_sequences(train_df, feature_cols)
print('Sequences:', X.shape, y.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 128
train_ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:
from scipy.stats import spearmanr

def sharpe_spearman(preds: pd.DataFrame, reals: pd.DataFrame):
    """Compute competition metric.
    preds & reals must have columns ['timestamp', 'target_id', 'pred'] / 'real'."""
    merged = preds.merge(reals, on=['timestamp', 'target_id'], how='inner')
    daily_rhos = []
    for ts, grp in merged.groupby('timestamp'):
        rho, _ = spearmanr(grp['pred'], grp['real'])
        daily_rhos.append(rho)
    rhos = np.array(daily_rhos)
    return rhos.mean() / rhos.std(ddof=0)


In [ ]:
model = TimeXer(
    h= PRED_LEN,
    input_size=len(feature_cols),
    seq_len=SEQ_LEN,
    d_model=64,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
).to('cuda' if torch.cuda.is_available() else 'cpu')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


In [ ]:
EPOCHS = 10
device = next(model.parameters()).device

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out.squeeze(), yb.squeeze())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    print(f"Epoch {epoch:02d}  train_loss={total_loss/len(train_loader.dataset):.6f}")
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
# TODO: create a proper rolling‑window validation that mirrors the competition.
# Example (pseudo):
# 1. Hold‑out last N days
# 2. Predict per day; collect preds & reals
# 3. Compute sharpe_spearman(preds, reals)

# val_score = sharpe_spearman(pred_df, real_df)
# print('Validation metric:', val_score)


In [ ]:
test_path = TEST_DIR/'test.parquet'
test_df = pd.read_parquet(test_path)
# Preprocess the same way (fill, scale)
test_df[feature_cols] = test_df[feature_cols].fillna(0)
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

# Predict per asset day‑by‑day
predictions = []
model.eval()
for tid, grp in tqdm(test_df.groupby('target_id')):
    grp = grp.sort_values('timestamp').reset_index(drop=True)
    feats = grp[feature_cols].values.astype(np.float32)
    seqs = torch.from_numpy(np.stack([feats[-SEQ_LEN:]])).to(device)  # last window
    with torch.no_grad():
        pred = model(seqs).cpu().numpy().flatten()[0]
    predictions.append((tid, grp.iloc[-1]['row_id'], pred))

pred_df = pd.DataFrame(predictions, columns=['target_id', 'row_id', 'target'])
submission = pred_df[['row_id', 'target']]
submission.to_csv('submission.csv', index=False)
print('Submission saved:', submission.shape)